In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# CHANGE THIS
DATASET_ROOT = Path("/content/drive/MyDrive/isaac_scene1/patched")

print("Dataset root exists:", DATASET_ROOT.exists())
print("Dataset root:", DATASET_ROOT)

In [ ]:
from pathlib import Path
import re

png_files = sorted(DATASET_ROOT.rglob("*.png"))
json_files = sorted(DATASET_ROOT.rglob("*.json"))
npy_files = sorted(DATASET_ROOT.rglob("*.npy"))

rgb_files = [p for p in png_files if "rgb_" in p.name.lower()]
bbox2d_json_files = [p for p in json_files if "bounding_box_2d_tight" in p.name.lower()]
bbox2d_label_json_files = [p for p in json_files if "bounding_box_2d_tight_labels" in p.name.lower()]
bbox2d_npy_files = [p for p in npy_files if "bounding_box_2d_tight" in p.name.lower()]

def extract_index(filename):
    m = re.search(r'_(\d+)', filename)
    return m.group(1) if m else None

rgb_map = {extract_index(p.stem): p for p in rgb_files if extract_index(p.stem)}
bbox_json_map = {extract_index(p.stem): p for p in bbox2d_json_files if extract_index(p.stem)}
bbox_label_map = {extract_index(p.stem): p for p in bbox2d_label_json_files if extract_index(p.stem)}
bbox_npy_map = {extract_index(p.stem): p for p in bbox2d_npy_files if extract_index(p.stem)}

print("RGB:", len(rgb_map))
print("2D bbox JSON:", len(bbox_json_map))
print("2D bbox label JSON:", len(bbox_label_map))
print("2D bbox NPY:", len(bbox_npy_map))

print("RGB sample files:")
for p in rgb_files[:10]:
    print(p)

print("\nJSON sample files:")
for p in json_files[:20]:
    print(p)

print("\nNPY sample files:")
for p in npy_files[:20]:
    print(p)

In [ ]:
import numpy as np

sample_npy = "/content/drive/MyDrive/isaac_scene1_patched/patched_dataset/bounding_box_2d_tight_0000.npy"
arr = np.load(sample_npy, allow_pickle=True)

print("TYPE:", type(arr))
print("SHAPE:", arr.shape)
print("DTYPE:", arr.dtype)
print("DTYPE NAMES:", arr.dtype.names)
print("\nFIRST 5 ROWS:")
print(arr[:5])

In [ ]:
import json
from pprint import pprint

sample_json = "/content/drive/MyDrive/isaac_scene1_patched/patched_dataset/bounding_box_2d_tight_labels_0000.json"

with open(sample_json, "r") as f:
    data = json.load(f)

print("TYPE:", type(data))
if isinstance(data, dict):
    print("KEYS:", list(data.keys()))
pprint(data)

In [ ]:
from pathlib import Path
import re

DATASET_ROOT = Path("/content/drive/MyDrive/isaac_scene1_patched/patched_dataset")

rgb_files = sorted(DATASET_ROOT.glob("rgb_*.png"))
npy_files = sorted(DATASET_ROOT.glob("bounding_box_2d_tight_*.npy"))
json_files = sorted(DATASET_ROOT.glob("bounding_box_2d_tight_labels_*.json"))

def extract_index(path_obj):
    # get the LAST _#### at the end of the stem
    m = re.search(r'_(\d+)$', path_obj.stem)
    return m.group(1) if m else None

rgb_map = {extract_index(p): p for p in rgb_files if extract_index(p) is not None}
npy_map = {extract_index(p): p for p in npy_files if extract_index(p) is not None}
json_map = {extract_index(p): p for p in json_files if extract_index(p) is not None}

common_indices = sorted(set(rgb_map.keys()) & set(npy_map.keys()) & set(json_map.keys()))

print("RGB:", len(rgb_map))
print("NPY:", len(npy_map))
print("JSON:", len(json_map))
print("Matched samples:", len(common_indices))
print("First 10 matched indices:", common_indices[:10])

In [ ]:
import numpy as np
import json

def normalize_class_name(name):
    name = name.strip().lower()
    replacements = {
        "cone,traffic_cone": "traffic_cone",
        "box,cardbox": "cardbox",
        "barel": "barrel",
    }
    return replacements.get(name, name)

def load_frame_objects(idx):
    npy_path = npy_map[idx]
    json_path = json_map[idx]

    arr = np.load(npy_path, allow_pickle=True)

    with open(json_path, "r") as f:
        label_map = json.load(f)

    objects = []
    for row in arr:
        semantic_id = int(row["semanticId"])
        x1 = int(row["x_min"])
        y1 = int(row["y_min"])
        x2 = int(row["x_max"])
        y2 = int(row["y_max"])
        occlusion = float(row["occlusionRatio"])

        class_name = label_map.get(str(semantic_id), {}).get("class", f"id_{semantic_id}")
        class_name = normalize_class_name(class_name)

        objects.append({
            "semantic_id": semantic_id,
            "class_name": class_name,
            "bbox": [x1, y1, x2, y2],
            "occlusion": occlusion,
        })

    return objects

In [ ]:
import cv2
import matplotlib.pyplot as plt

def draw_boxes_on_image(idx, selected_classes=None, min_box_size=3):
    img_path = rgb_map[idx]
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    objects = load_frame_objects(idx)

    if selected_classes is not None:
        selected_classes = set(c.strip().lower() for c in selected_classes)
        objects = [o for o in objects if o["class_name"] in selected_classes]

    shown = 0
    for obj in objects:
        x1, y1, x2, y2 = obj["bbox"]
        w = x2 - x1
        h = y2 - y1
        if w < min_box_size or h < min_box_size:
            continue

        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(
            image,
            obj["class_name"],
            (x1, max(y1 - 6, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 0),
            1,
            cv2.LINE_AA
        )
        shown += 1

    plt.figure(figsize=(16, 9))
    plt.imshow(image)
    plt.title(f"Frame {idx} | shown boxes: {shown}")
    plt.axis("off")
    plt.show()

In [ ]:
draw_boxes_on_image(common_indices[0], min_box_size=5)

In [ ]:
EXCLUDED_CLASSES = {
    "wall",
    "floor",
    "floor_decal",
    "pillar",
    "rack",
    "klt_bin",
    "sign"
}

In [ ]:
import numpy as np
import json

def normalize_class_name(name):
    name = name.strip().lower()
    replacements = {
        "cone,traffic_cone": "traffic_cone",
        "box,cardbox": "cardbox",
        "barel": "barrel",
    }
    return replacements.get(name, name)

def load_frame_objects(idx):
    npy_path = npy_map[idx]
    json_path = json_map[idx]

    arr = np.load(npy_path, allow_pickle=True)

    with open(json_path, "r") as f:
        label_map = json.load(f)

    objects = []
    for row in arr:
        semantic_id = int(row["semanticId"])
        x1 = int(row["x_min"])
        y1 = int(row["y_min"])
        x2 = int(row["x_max"])
        y2 = int(row["y_max"])
        occlusion = float(row["occlusionRatio"])

        class_name = label_map.get(str(semantic_id), {}).get("class", f"id_{semantic_id}")
        class_name = normalize_class_name(class_name)

        objects.append({
            "semantic_id": semantic_id,
            "class_name": class_name,
            "bbox": [x1, y1, x2, y2],
            "occlusion": occlusion,
        })

    return objects

In [ ]:
from collections import Counter

class_counter = Counter()

for idx in common_indices:
    for obj in load_frame_objects(idx):
        x1, y1, x2, y2 = obj["bbox"]
        if (x2 - x1) > 0 and (y2 - y1) > 0:
            class_counter[obj["class_name"]] += 1

print("Classes in dataset:")
for cls, count in class_counter.most_common():
    print(f"{cls}: {count}")

In [ ]:
all_classes = sorted(class_counter.keys())
final_classes = [c for c in all_classes if c not in EXCLUDED_CLASSES]

print("Excluded classes:")
for c in sorted(EXCLUDED_CLASSES):
    print("-", c)

print("\nFinal classes kept:")
for i, c in enumerate(final_classes):
    print(i, c)

CLASS_MAP = {cls_name: i for i, cls_name in enumerate(final_classes)}

In [ ]:
import cv2
import matplotlib.pyplot as plt

def draw_boxes_on_image(idx, excluded_classes=None, min_box_size=3):
    if excluded_classes is None:
        excluded_classes = set()

    excluded_classes = set(normalize_class_name(c) for c in excluded_classes)

    img_path = rgb_map[idx]
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    objects = load_frame_objects(idx)

    shown = 0
    for obj in objects:
        cls = obj["class_name"]
        if cls in excluded_classes:
            continue

        x1, y1, x2, y2 = obj["bbox"]
        w = x2 - x1
        h = y2 - y1

        if w < min_box_size or h < min_box_size:
            continue

        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(
            image,
            cls,
            (x1, max(y1 - 6, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 0),
            1,
            cv2.LINE_AA
        )
        shown += 1

    plt.figure(figsize=(16, 9))
    plt.imshow(image)
    plt.title(f"Frame {idx} | shown boxes: {shown}")
    plt.axis("off")
    plt.show()

In [ ]:
draw_boxes_on_image(common_indices[0], excluded_classes=EXCLUDED_CLASSES, min_box_size=5)

In [ ]:
import shutil
from PIL import Image
from pathlib import Path

YOLO_ROOT = DATASET_ROOT / "yolo_filtered"
IMAGES_DIR = YOLO_ROOT / "images" / "train"
LABELS_DIR = YOLO_ROOT / "labels" / "train"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

def clamp(val, lo, hi):
    return max(lo, min(val, hi))

def xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h):
    x_center = ((x1 + x2) / 2.0) / img_w
    y_center = ((y1 + y2) / 2.0) / img_h
    width = (x2 - x1) / img_w
    height = (y2 - y1) / img_h
    return x_center, y_center, width, height

converted = 0
skipped_empty = 0
total_boxes_written = 0

excluded_classes = set(normalize_class_name(c) for c in EXCLUDED_CLASSES)

for idx in common_indices:
    img_path = rgb_map[idx]
    objects = load_frame_objects(idx)

    with Image.open(img_path) as img:
        img_w, img_h = img.size

    yolo_lines = []

    for obj in objects:
        class_name = obj["class_name"]

        if class_name in excluded_classes:
            continue

        if class_name not in CLASS_MAP:
            continue

        x1, y1, x2, y2 = obj["bbox"]

        x1 = clamp(x1, 0, img_w - 1)
        y1 = clamp(y1, 0, img_h - 1)
        x2 = clamp(x2, 0, img_w - 1)
        y2 = clamp(y2, 0, img_h - 1)

        w = x2 - x1
        h = y2 - y1

        if w < 3 or h < 3:
            continue

        xc, yc, bw, bh = xyxy_to_yolo(x1, y1, x2, y2, img_w, img_h)

        if bw <= 0 or bh <= 0:
            continue

        class_id = CLASS_MAP[class_name]
        yolo_lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    if len(yolo_lines) == 0:
        skipped_empty += 1
        continue

    shutil.copy2(img_path, IMAGES_DIR / img_path.name)

    with open(LABELS_DIR / f"{img_path.stem}.txt", "w") as f:
        f.write("\n".join(yolo_lines))

    converted += 1
    total_boxes_written += len(yolo_lines)

print("Converted images:", converted)
print("Skipped empty images:", skipped_empty)
print("Total boxes written:", total_boxes_written)
print("Saved to:", YOLO_ROOT)

In [ ]:
yaml_path = YOLO_ROOT / "data.yaml"

with open(yaml_path, "w") as f:
    f.write(f"path: {YOLO_ROOT}\n")
    f.write("train: images/train\n")
    f.write("val: images/train\n\n")
    f.write("names:\n")
    for i, cls_name in enumerate(final_classes):
        f.write(f"  {i}: {cls_name}\n")

print(yaml_path.read_text())

In [ ]:
from pathlib import Path
import random
import shutil

YOLO_ROOT = Path("/content/drive/MyDrive/isaac_scene1/patched/yolo_filtered")

train_img_dir = YOLO_ROOT / "images" / "train"
train_lbl_dir = YOLO_ROOT / "labels" / "train"
val_img_dir   = YOLO_ROOT / "images" / "val"
val_lbl_dir   = YOLO_ROOT / "labels" / "val"
test_img_dir  = YOLO_ROOT / "images" / "test"
test_lbl_dir  = YOLO_ROOT / "labels" / "test"

# create missing folders
for d in [val_img_dir, val_lbl_dir, test_img_dir, test_lbl_dir]:
    d.mkdir(parents=True, exist_ok=True)

# safety check: only split if val and test are empty
if any(val_img_dir.glob("*.png")) or any(test_img_dir.glob("*.png")):
    print("Val/test are not empty. Move files back first before splitting again.")
else:
    image_files = sorted(train_img_dir.glob("*.png"))
    print("Images currently in train:", len(image_files))

    random.seed(42)
    random.shuffle(image_files)

    val_ratio = 0.2
    test_ratio = 0.2

    num_total = len(image_files)
    num_val = int(num_total * val_ratio)
    num_test = int(num_total * test_ratio)

    val_files = image_files[:num_val]
    test_files = image_files[num_val:num_val + num_test]

    # move val
    for img_path in val_files:
        lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
        shutil.move(str(img_path), str(val_img_dir / img_path.name))
        if lbl_path.exists():
            shutil.move(str(lbl_path), str(val_lbl_dir / lbl_path.name))

    # move test
    for img_path in test_files:
        lbl_path = train_lbl_dir / f"{img_path.stem}.txt"
        shutil.move(str(img_path), str(test_img_dir / img_path.name))
        if lbl_path.exists():
            shutil.move(str(lbl_path), str(test_lbl_dir / lbl_path.name))

    print(f"Moved {len(val_files)} images to val.")
    print(f"Moved {len(test_files)} images to test.")
    print(f"Remaining in train: {len(list(train_img_dir.glob('*.png')))}")

In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path

YOLO_ROOT = Path("/content/drive/MyDrive/isaac_scene1/yolo_filtered")

for split in ["train", "val", "test"]:
    img_dir = YOLO_ROOT / "images" / split
    lbl_dir = YOLO_ROOT / "labels" / split
    print(f"{split}: images={len(list(img_dir.glob('*.png')))}, labels={len(list(lbl_dir.glob('*.txt')))}")

print("\ndata.yaml exists:", (YOLO_ROOT / "data.yaml").exists())
print((YOLO_ROOT / "data.yaml").read_text())

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data="/content/drive/MyDrive/isaac_scene1/yolo_filtered/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    workers=2,
    project="/content/drive/MyDrive/isaac_scene1/yolo_runs",
    name="yolov8n_isaac_scene1",
    pretrained=True,
    cache=False,
    patience=20,
    verbose=True
)

In [ ]:
metrics_val = model.val(
    data="/content/drive/MyDrive/isaac_scene1/yolo_filtered/data.yaml",
    split="val"
)
print(metrics_val)

In [ ]:
from pathlib import Path

yaml_path = Path("/content/drive/MyDrive/isaac_scene1/yolo_filtered/data.yaml")
print("Exists:", yaml_path.exists())
print(yaml_path.read_text())

In [ ]:
DATA_YAML = "/content/drive/MyDrive/isaac_scene1/yolo_filtered/data.yaml"

In [ ]:
from pathlib import Path

YOLO_ROOT = Path("/content/drive/MyDrive/isaac_scene1/yolo_filtered")
yaml_path = YOLO_ROOT / "data.yaml"

yaml_text = f"""path: {YOLO_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: barrel
  1: box
  2: bucket
  3: cardbox
  4: crate
  5: fire_extinguisher
  6: forklift
  7: pallet
  8: person
  9: sign
  10: traffic_cone
"""

yaml_path.write_text(yaml_text)
print(yaml_path.read_text())

In [ ]:
from pathlib import Path

YOLO_ROOT = Path("/content/drive/MyDrive/isaac_scene1/yolo_filtered")

for split in ["train", "val", "test"]:
    img_dir = YOLO_ROOT / "images" / split
    lbl_dir = YOLO_ROOT / "labels" / split
    print(f"{split}:")
    print("  image dir exists:", img_dir.exists(), img_dir)
    print("  label dir exists:", lbl_dir.exists(), lbl_dir)
    print("  images:", len(list(img_dir.glob("*.png"))))
    print("  labels:", len(list(lbl_dir.glob("*.txt"))))

In [ ]:
from pathlib import Path

test_images = sorted((YOLO_ROOT / "images" / "test").glob("*.png"))[:5]

best_model = YOLO("/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene1/weights/best.pt")

for img_path in test_images:
    print("Running on:", img_path.name)
    results = best_model.predict(
        source=str(img_path),
        imgsz=640,
        conf=0.25,
        save=True,
        project="/content/drive/MyDrive/isaac_scene1/yolo_predictions",
        name="test_preds"
    )